<a href="https://colab.research.google.com/github/shehan16/Machine-Learning-and-AI/blob/main/Multi_National_Company_Finder_Agent_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install crewai_tools crewai

In [ ]:
from crewai import Agent, Task, Crew, LLM
#from langchain.agents import load_tools
from crewai.tools import tool
import requests
import yfinance as yf
import json
from crewai_tools import SerperDevTool, WebsiteSearchTool, ScrapeWebsiteTool
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Dict, Optional, List, Set, Tuple
import pandas as pd

# Import userdata for Colab secrets
from google.colab import userdata

# API Keys - Retrieved from Colab Secrets
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
SERPER_API_KEY = userdata.get("SERPER_API_KEY")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

### Configure the LLM

In [ ]:
llm_openai = LLM(
model='gpt-4o-mini',
#model='gpt-4o',
temperature=0,
base_url="https://api.openai.com/v1",
api_key= OPENAI_API_KEY
)


In [ ]:
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY
os.environ['SERPER_API_KEY']=SERPER_API_KEY

### Create the Pydantic Class for Structured Output from Agent

In [ ]:
# ----------- Define Pydantic Models for Structured Output -----------

class AllInformation(BaseModel):
    company_name:str = Field(...,description="The name of the company")
    company_website:str = Field(...,description="The website of the company")
    domain:str = Field(...,description="The domain of the company")
    countries_operating: List[str] = Field(...,description="List of countries where the company's offices are located.")
    is_multinational: str = Field(...,description="Indicates whether the company's offices are located in multiple countries. Return Yes, No or Unknown. If UAE is the only country the company is operating in, say No")
    why_multinational: str = Field(...,description="A short explanation of reason behind why you decided a company as multinational or not")


class AllInformationList(BaseModel):
    information: List[AllInformation] = Field(..., description="A list of companies with information extracted")

### Define Agents

In [ ]:
data_collection_agent = Agent(
    role="Market Research Specialist",
    goal=(
        "Find the official website for the company {COMPANY}"
    ),
    backstory=(
        "This company is confirmed to have operations in the UAE. As it is assumed to be a "
        "Small or Medium-sized Business (SMB), there may be limited information available online. "
        "Prioritize extracting data from reliable sources while considering the possibility of sparse details."
    ),
    verbose=True,  # Enable detailed logging for better traceability.
    allow_delegation=False,  # Ensure tasks are handled directly without delegation.
    llm=llm_openai,  # Specify the language model to be used (OpenAI in this case).
    max_iter=15,  # Limit the number of iterations to avoid excessive processing.
    memory=True,  # Enable memory for contextual continuity during task execution.
    tools=[
        SerperDevTool(),  # Tool for advanced search capabilities.
        ScrapeWebsiteTool(),  # Tool for scraping relevant data from websites.
        WebsiteSearchTool()  # Tool for general website searches.
    ],
)

In [ ]:
data_extraction_agent = Agent(
    role="Professional Data Analyst",
    goal=(
        "Find the official website for the company {COMPANY}"
        "Extract detailedinformation from the <About Us> and <Contact Us> sections in the compan's website, to locate office locations of the company."
        "Ensure all extracted information is accurate by double-checking your findings. Avoid making assumptions or guesses."
    ),
    backstory=(
        "This company is confirmed to have operations in the UAE. As it is assumed to be a "
        "Small or Medium-sized Business (SMB), there may be limited information available online."
    ),
    verbose=True,  # Enable detailed logging for better traceability.
    allow_delegation=True,  # Allow delegation to tools or other agents for enhanced efficiency.
    llm=llm_openai,  # Specify the language model to be used (OpenAI in this case).
    max_iter=10,  # Limit the number of iterations to ensure efficient task execution.
    memory=True,  # Enable memory for contextual continuity during task execution.
    tools=[
        SerperDevTool(),  # Tool for advanced search capabilities.
        ScrapeWebsiteTool(),  # Tool for extracting data from websites.
        WebsiteSearchTool()  # Tool for general website searches.
    ],
)

### Define Tasks

In [ ]:
data_collection_task = Task(
    description=(
        "Perform a web search to locate the official website of the company {COMPANY}. "
        "If the official website cannot be located, use alternative resources such as 2gis.ae or "
        "Yellowpages UAE to find information about the company or its website. "
        "Exercise caution to ensure that only the correct website associated with the company {COMPANY} "
        "is collected, as some company names may be similar to others."
    ),
    expected_output="A compiled set of accurate and verified information about the company {COMPANY}.",
    tools=[
        SerperDevTool(),  # Tool for advanced search capabilities.
        ScrapeWebsiteTool()  # Tool for extracting data from websites.
    ],
    agent=data_collection_agent,  # Assign the data collection agent to handle the task.
)

In [ ]:
extract_information_task = Task(
    description=(
        "Extract the following information about the company {COMPANY}, which has operations in the UAE. "
        "Ensure you specifically search the <About Us> and <Contact Us> sections on the company's website. "
        "- The name of the company.\n"
        "- The company website (If the website cannot be directly found via web searching, check LinkedIn or 2gis.ae).\n"
        "- The domain area of the company.\n"
        "- A list of country names where the company's offices are located. Extract country names only, not regions or other details. "
        "UAE is definitely included. Use the <About Us> or <Contact Us> section to confirm if the company operates in other countries. Be specific when mentioning countries.\n"
        "- Indicate whether the company's offices are in multiple countries. Return 'Yes', 'No', or 'NA'. If UAE is the only country the company operates in, or if you are unsure, return 'No'.\n\n"
        "If any information cannot be found for a specific field, return 'NA'."
    ),
    agent=data_extraction_agent,  # Assign the data extraction agent to handle the task.
    expected_output=(
        "A tabular output with extracted answers for the specified variables, formatted according to the pydantic class."
    ),
    output_pydantic=AllInformationList,  # Specify the pydantic class for structured output.
    context=[data_collection_task],  # Provide context from the data collection task.
)

### Define Crew

In [ ]:
crew = Crew(
   agents=[data_collection_agent,data_extraction_agent],
    tasks=[data_collection_task,extract_information_task],
    #verbose=True,
    #process=Process.sequential,
    full_output=True,
    max_iter=10,
)

### Final Function to determine Multi National Company Flag

In [ ]:
def extract_company_info(company):
    inputs = {
            'COMPANY': company,
            }

    results = crew.kickoff(inputs=inputs)
    df1 = pd.DataFrame(results.pydantic.dict()['information'])
    info_json = df1.to_dict(orient="records")
    return info_json

In [ ]:
extract_company_info('e& UAE')

In [ ]:
extract_company_info('VERSOL MIDDLE EAST DWC LLC')

In [ ]:
company= 'RASHID AL JABRI GROUP'

inputs = {
  'COMPANY': company,
}

results = crew.kickoff(inputs=inputs)
results.pydantic.dict()['information']

In [ ]:
company= 'Mohinani Group'

inputs = {
  'COMPANY': company,
}

results = crew.kickoff(inputs=inputs)
results.pydantic.dict()